In [1]:
import pandas as pd

# Path to your SQL file
file_path = "G:\VSCODE\Python And Data Sceince\Sql\ipl_db.sql"

# Read file
with open(file_path, "r", encoding="utf-8") as file:
    sql_content = file.read()

print("File loaded successfully!")
print("Length of file:", len(sql_content))

File loaded successfully!
Length of file: 44637129


In [2]:
import re
import pandas as pd
from io import StringIO

# Extract deliveries COPY block
pattern = r"COPY public\.deliveries \((.*?)\) FROM stdin;\n(.*?)\n\\\."
match = re.search(pattern, sql_content, re.DOTALL)

columns = match.group(1).split(", ")
data = match.group(2)

# Convert to DataFrame
deliveries_df = pd.read_csv(
    StringIO(data),
    sep="\t",
    names=columns,
    na_values="\\N"
)

print("Deliveries shape:", deliveries_df.shape)
deliveries_df.head()

Deliveries shape: (278171, 22)


,match_id,inning,over,ball,over_ball,batting_team,bowling_team,batsman,non_striker,bowler,...,is_wide,is_no_ball,byes,leg_byes,penalty,dismissal_kind,player_dismissed,date,batsman_type,bowler_type
0,335982,1,0,6,0.6,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,SC Ganguly,P Kumar,...,f,f,f,f,f,NaN,NaN,2008-04-18,Right hand Bat,Right arm Medium
1,335982,1,2,6,2.6,Kolkata Knight Riders,Royal Challengers Bangalore,SC Ganguly,BB McCullum,P Kumar,...,f,f,f,f,f,NaN,NaN,2008-04-18,Left hand Bat,Right arm Medium
2,335982,1,9,5,9.5,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,RT Ponting,SB Joshi,...,f,f,f,f,f,NaN,NaN,2008-04-18,Right hand Bat,Slow Left arm Orthodox
3,335982,1,16,6,16.6,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,DJ Hussey,Z Khan,...,f,f,f,f,f,NaN,NaN,2008-04-18,Right hand Bat,Left arm Fast medium
4,335982,1,17,6,17.6,Kolkata Knight Riders,Royal Challengers Bangalore,BB McCullum,Mohammad Hafeez,AA Noffke,...,f,f,f,f,f,NaN,NaN,2008-04-18,Right hand Bat,Right arm Fast medium


In [3]:
# Extract matches COPY block
pattern_matches = r"COPY public\.matches \((.*?)\) FROM stdin;\n(.*?)\n\\\."
match_matches = re.search(pattern_matches, sql_content, re.DOTALL)

columns_matches = match_matches.group(1).split(", ")
data_matches = match_matches.group(2)

# Convert to DataFrame
matches_df = pd.read_csv(
    StringIO(data_matches),
    sep="\t",
    names=columns_matches,
    na_values="\\N"
)

print("Matches shape:", matches_df.shape)
matches_df.head()

Matches shape: (1169, 28)


,match_id,season,date,venue,city,event,team1,team2,toss_winner,toss_decision,...,gender,balls_per_over,match_number,umpire1,umpire2,tv_umpire,reserve_umpire,match_referee,year,match_stage
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Indian Premier League,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,...,male,6,1.0,AY Dandekar,NJ Llong,A Deshmukh,N Pandit,J Srinath,2017,League
1,1082592,2017,2017-04-06,Maharashtra Cricket Association Stadium,Pune,Indian Premier League,Rising Pune Supergiant,Mumbai Indians,Rising Pune Supergiant,field,...,male,6,2.0,A Nand Kishore,S Ravi,VK Sharma,Navdeep Singh,M Nayyar,2017,League
2,1082593,2017,2017-04-07,Saurashtra Cricket Association Stadium,Rajkot,Indian Premier League,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,...,male,6,3.0,Nitin Menon,CK Nandan,YC Barde,K Srinivasan,V Narayan Kutty,2017,League
3,1082594,2017,2017-04-08,Holkar Cricket Stadium,Indore,Indian Premier League,Kings XI Punjab,Rising Pune Supergiant,Kings XI Punjab,field,...,male,6,4.0,AK Chaudhary,C Shamshuddin,KN Ananthapadmanabhan,R Pandit,Chinmay Sharma,2017,League
4,1082595,2017,2017-04-08,M.Chinnaswamy Stadium,Bengaluru,Indian Premier League,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,...,male,6,5.0,S Ravi,VK Sharma,A Nand Kishore,Navdeep Singh,J Srinath,2017,League


In [4]:
# Merge datasets
df = deliveries_df.merge(
    matches_df[['match_id', 'winner']],
    on='match_id',
    how='left'
)

print("Merged shape:", df.shape)

# Create target column
df['batting_team_won'] = (df['batting_team'] == df['winner']).astype(int)

# Quick check
df[['batting_team', 'winner', 'batting_team_won']].head(10)

Merged shape: (278171, 23)


,batting_team,winner,batting_team_won
0,Kolkata Knight Riders,Kolkata Knight Riders,1
1,Kolkata Knight Riders,Kolkata Knight Riders,1
2,Kolkata Knight Riders,Kolkata Knight Riders,1
3,Kolkata Knight Riders,Kolkata Knight Riders,1
4,Kolkata Knight Riders,Kolkata Knight Riders,1
5,Kolkata Knight Riders,Kolkata Knight Riders,1
6,Royal Challengers Bangalore,Kolkata Knight Riders,0
7,Sunrisers Hyderabad,Sunrisers Hyderabad,1
8,Sunrisers Hyderabad,Sunrisers Hyderabad,1
9,Sunrisers Hyderabad,Sunrisers Hyderabad,1


In [5]:
# Convert boolean columns properly
df['is_wide'] = df['is_wide'].map({'t': 1, 'f': 0})
df['is_no_ball'] = df['is_no_ball'].map({'t': 1, 'f': 0})

# Total runs per ball (batsman + extras)
df['total_ball_runs'] = df['batsman_runs'] + df['extras']

# Sort properly
df = df.sort_values(by=['match_id', 'inning', 'over', 'ball'])

# Cumulative runs
df['total_runs'] = df.groupby(['match_id', 'inning'])['total_ball_runs'].cumsum()

# Wickets fallen
df['is_wicket'] = df['player_dismissed'].notna().astype(int)
df['wickets_fallen'] = df.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

# Balls bowled
df['balls_bowled'] = df.groupby(['match_id', 'inning']).cumcount() + 1

# Current run rate
df['current_run_rate'] = df['total_runs'] / (df['balls_bowled'] / 6)

# Balls remaining (T20 = 120 balls)
df['balls_remaining'] = 120 - df['balls_bowled']

# Preview
df[['total_runs', 'wickets_fallen', 'balls_bowled', 'current_run_rate', 'balls_remaining']].head(15)

,total_runs,wickets_fallen,balls_bowled,current_run_rate,balls_remaining
260743,1,0,1,6.000000,119
69431,1,0,2,3.000000,118
260744,2,0,3,4.000000,117
17345,2,0,4,3.000000,116
34765,2,0,5,2.400000,115
0,2,0,6,2.000000,114
139154,3,0,7,2.571429,113
139155,3,0,8,2.250000,112
52197,7,0,9,4.666667,111
139156,11,0,10,6.600000,110


In [6]:
# Step 1: Get total runs of 1st innings (target calculation)
first_innings = df[df['inning'] == 1]

targets = first_innings.groupby('match_id')['total_runs'].max().reset_index()
targets.rename(columns={'total_runs': 'first_innings_score'}, inplace=True)

# Step 2: Merge target into full dataframe
df = df.merge(targets, on='match_id', how='left')

# Step 3: Create target runs (add 1)
df['target_runs'] = df['first_innings_score'] + 1

# Step 4: Focus on 2nd innings only
df_2nd = df[df['inning'] == 2].copy()

# Step 5: Calculate chase features
df_2nd['runs_remaining'] = df_2nd['target_runs'] - df_2nd['total_runs']

df_2nd['required_run_rate'] = df_2nd['runs_remaining'] / (df_2nd['balls_remaining'] / 6)

# Pressure index
df_2nd['pressure_index'] = df_2nd['required_run_rate'] / df_2nd['current_run_rate']

# Preview
df_2nd[['total_runs', 'target_runs', 'runs_remaining', 
        'current_run_rate', 'required_run_rate', 'pressure_index']].head(15)

,total_runs,target_runs,runs_remaining,current_run_rate,required_run_rate,pressure_index
124,1,223,222,6.000000,11.193277,1.865546
125,2,223,221,6.000000,11.237288,1.872881
126,2,223,221,4.000000,11.333333,2.833333
127,3,223,220,4.500000,11.379310,2.528736
128,4,223,219,4.800000,11.426087,2.380435
129,4,223,219,4.000000,11.526316,2.881579
130,4,223,219,3.428571,11.628319,3.391593
131,4,223,219,3.000000,11.732143,3.910714
132,4,223,219,2.666667,11.837838,4.439189
133,8,223,215,4.800000,11.727273,2.443182


In [7]:
# Select features
features = [
    'total_runs',
    'wickets_fallen',
    'balls_remaining',
    'current_run_rate',
    'required_run_rate',
    'pressure_index'
]

# Drop NaNs (important for safety)
df_model = df_2nd.dropna(subset=features + ['batting_team_won'])

# Define X and y
X = df_model[features]
y = df_model['batting_team_won']

print("Final dataset shape:", X.shape)
X.head()

Final dataset shape: (133876, 6)


,total_runs,wickets_fallen,balls_remaining,current_run_rate,required_run_rate,pressure_index
124,1,0,119,6.0,11.193277,1.865546
125,2,0,118,6.0,11.237288,1.872881
126,2,0,117,4.0,11.333333,2.833333
127,3,0,116,4.5,11.379310,2.528736
128,4,0,115,4.8,11.426087,2.380435


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss

In [11]:
import numpy as np

# Replace inf values with NaN
df_model.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with NaN
df_model.dropna(inplace=True)

# Recreate X and y
X = df_model[features]
y = df_model['batting_team_won']

print("Cleaned dataset shape:", X.shape)

Cleaned dataset shape: (6358, 6)


C:\Users\sharm\AppData\Local\Temp\ipykernel_3068\3769048520.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\sharm\AppData\Local\Temp\ipykernel_3068\3769048520.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_model.dropna(inplace=True)


In [12]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log Loss:", log_loss(y_test, y_prob))

Accuracy: 0.7161949685534591
Log Loss: 0.5280821569791914


In [13]:
def predict_win_probability(
    total_runs,
    wickets_fallen,
    balls_remaining,
    current_run_rate,
    required_run_rate,
    pressure_index
):
    import numpy as np
    
    input_data = np.array([[
        total_runs,
        wickets_fallen,
        balls_remaining,
        current_run_rate,
        required_run_rate,
        pressure_index
    ]])
    
    prob = model.predict_proba(input_data)[0][1]
    
    return prob

In [14]:
predict_win_probability(
    total_runs=120,
    wickets_fallen=3,
    balls_remaining=60,
    current_run_rate=8.0,
    required_run_rate=7.5,
    pressure_index=0.93
)

c:\Users\sharm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


np.float64(0.5206414618991269)

In [15]:
from xgboost import XGBClassifier

# Initialize model
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate
from sklearn.metrics import accuracy_score, log_loss

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost Log Loss:", log_loss(y_test, y_prob_xgb))

XGBoost Accuracy: 0.7940251572327044
XGBoost Log Loss: 0.42344261215925694


c:\Users\sharm\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:200: UserWarning: [12:03:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [16]:
def predict_win_probability_xgb(
    total_runs,
    wickets_fallen,
    balls_remaining,
    current_run_rate,
    required_run_rate,
    pressure_index
):
    import numpy as np
    
    input_data = np.array([[
        total_runs,
        wickets_fallen,
        balls_remaining,
        current_run_rate,
        required_run_rate,
        pressure_index
    ]])
    
    prob = xgb_model.predict_proba(input_data)[0][1]
    
    return float(prob)

In [17]:
predict_win_probability_xgb(
    total_runs=120,
    wickets_fallen=3,
    balls_remaining=60,
    current_run_rate=8.0,
    required_run_rate=7.5,
    pressure_index=0.93
)

0.6937741041183472

In [19]:
import joblib

# Save model
joblib.dump(xgb_model, "ipl_win_predictor_xgb.pkl")

print("Model saved successfully!")

Model saved successfully!
